# LeetCode 1344 — Angle Between Hands of a Clock bằng Python

Link đề: https://leetcode.com/problems/angle-between-hands-of-a-clock/ · Độ khó: **Medium** · Chủ đề: Math

Skeleton LeetCode:

```python
class Solution:
    def angleClock(self, hour: int, minutes: int) -> float:
        ...
```

## 1. Mục tiêu học

1. Mô hình hóa một bài toán đời thường (đồng hồ kim) thành **toán học về góc**: mỗi kim quay bao nhiêu độ mỗi phút.
2. Hiểu chỗ "bẫy" của bài: **kim giờ cũng di chuyển theo phút** (0,5° mỗi phút), không đứng yên ở vạch giờ.
3. Đi từ **mô phỏng từng phút** (baseline) → **công thức góc từng kim** → **công thức gộp** `|30h − 5.5m|` → phiên bản **số nguyên / phân số chính xác**.
4. Biết lấy **góc nhỏ hơn** bằng `min(x, 360 − x)` và xử lý `hour = 12` bằng `% 12`.
5. Biết cách **so sánh số thực** an toàn (`math.isclose`, sai số `1e-5`) và kiểm chứng bằng `fractions.Fraction`.

## 2. Đề bài & ví dụ

Cho hai số nguyên `hour` (giờ) và `minutes` (phút) đọc trên một đồng hồ kim. Trả về **góc nhỏ hơn** (tính bằng độ) tạo bởi **kim giờ** và **kim phút**. Đáp án sai lệch trong phạm vi `10^-5` so với giá trị thật được chấp nhận.

**Ví dụ 1:** `hour = 12, minutes = 30` → `165`

**Ví dụ 2:** `hour = 3, minutes = 30` → `75`

**Ví dụ 3:** `hour = 3, minutes = 15` → `7.5`

**Ràng buộc:**
- `1 <= hour <= 12`
- `0 <= minutes <= 59`

Nhận xét nhanh: chỉ có `12 × 60 = 720` bộ input khác nhau ⇒ ta có thể **kiểm tra toàn bộ không gian input** trong bộ test.

In [1]:
import math
import random
from fractions import Fraction

examples = [
    (12, 30, 165.0),
    (3, 30, 75.0),
    (3, 15, 7.5),
]
for h, m, want in examples:
    print(f"hour={h:2d}, minutes={m:2d} -> mong đợi {want}")
print("Số bộ input khác nhau:", 12 * 60)

hour=12, minutes=30 -> mong đợi 165.0
hour= 3, minutes=30 -> mong đợi 75.0
hour= 3, minutes=15 -> mong đợi 7.5
Số bộ input khác nhau: 720


## 3. Kiến thức nền

### 3.1 Vòng tròn 360° và tốc độ quay của từng kim

- Mặt đồng hồ là một vòng tròn **360°**, chia thành **12 vạch giờ** ⇒ mỗi vạch giờ cách nhau `360 / 12 = 30°`.
- **Kim phút** quay hết 1 vòng trong 60 phút ⇒ `360 / 60 = 6°` mỗi phút.
- **Kim giờ** quay hết 1 vòng trong 12 giờ = 720 phút ⇒ `360 / 720 = 0.5°` mỗi phút, tức `30°` mỗi giờ.

Ta đo mọi góc **theo chiều kim đồng hồ, bắt đầu từ vạch 12** (vị trí 0°).

In [2]:
DEG_PER_HOUR_MARK = 360 / 12        # khoảng cách giữa 2 vạch giờ
MINUTE_HAND_PER_MIN = 360 / 60      # kim phút: độ / phút
HOUR_HAND_PER_MIN = 360 / (12 * 60) # kim giờ: độ / phút
print("Mỗi vạch giờ:", DEG_PER_HOUR_MARK, "độ")
print("Kim phút:", MINUTE_HAND_PER_MIN, "độ/phút")
print("Kim giờ :", HOUR_HAND_PER_MIN, "độ/phút  =", HOUR_HAND_PER_MIN * 60, "độ/giờ")

Mỗi vạch giờ: 30.0 độ
Kim phút: 6.0 độ/phút
Kim giờ : 0.5 độ/phút  = 30.0 độ/giờ


### 3.2 Phép chia lấy dư `%` — đưa 12 giờ về 0

Trên mặt đồng hồ, "12 giờ" chính là vị trí **0°** (cùng vạch với 0 giờ). Dùng `hour % 12` để quy `12 → 0`, các giờ khác giữ nguyên. Tương tự, góc `x % 360` đưa mọi góc về đoạn `[0, 360)`.

In [3]:
for h in [1, 3, 11, 12]:
    print(f"hour={h:2d} -> hour % 12 = {h % 12}")
for x in [0, 90, 360, 375, -30]:
    print(f"{x:4d} % 360 = {x % 360}")   # chú ý: Python cho kết quả không âm với số chia dương

hour= 1 -> hour % 12 = 1
hour= 3 -> hour % 12 = 3
hour=11 -> hour % 12 = 11
hour=12 -> hour % 12 = 0
   0 % 360 = 0
  90 % 360 = 90
 360 % 360 = 0
 375 % 360 = 15
 -30 % 360 = 330


### 3.3 `abs` và `min` — khoảng cách giữa hai góc và góc nhỏ hơn

Hai kim ở vị trí `a` và `b` (độ). Hiệu `|a − b|` là một trong hai cung giữa chúng; cung còn lại là `360 − |a − b|`. Đề hỏi **góc nhỏ hơn** ⇒ `min(d, 360 − d)`.

In [4]:
a, b = 15.0, 195.0          # hai kim cách nhau nửa vòng
d = abs(a - b)
print("d =", d, "| cung còn lại =", 360 - d, "| góc nhỏ hơn =", min(d, 360 - d))
a, b = 350.0, 10.0          # hai kim gần nhau qua vạch 12
d = abs(a - b)
print("d =", d, "| cung còn lại =", 360 - d, "| góc nhỏ hơn =", min(d, 360 - d))

d = 180.0 | cung còn lại = 180.0 | góc nhỏ hơn = 180.0
d = 340.0 | cung còn lại = 20.0 | góc nhỏ hơn = 20.0


### 3.4 Số thực (float) và sai số

`float` trong Python là số dấu phẩy động 64-bit; nhiều số thập phân (như `0.1`) **không biểu diễn chính xác**. Vì vậy **không so sánh float bằng `==`**, mà dùng `math.isclose(a, b, abs_tol=1e-9)` hoặc `abs(a − b) < 1e-5` như đề cho phép.

May mắn ở bài này: mọi góc đều là **bội của 0.5** (vì kim giờ đi 0.5°/phút, kim phút đi 6°/phút) nên float biểu diễn chính xác — nhưng thói quen so sánh có sai số vẫn rất nên giữ.

In [5]:
print(0.1 + 0.2 == 0.3, 0.1 + 0.2)
print(math.isclose(0.1 + 0.2, 0.3, abs_tol=1e-9))
print("5.5 * 15 =", 5.5 * 15, "| bội của 0.5 nên chính xác:", (5.5 * 15) * 2 == int((5.5 * 15) * 2))

False 0.30000000000000004
True
5.5 * 15 = 82.5 | bội của 0.5 nên chính xác: True


### 3.5 `fractions.Fraction` — số hữu tỉ chính xác

`Fraction(p, q)` lưu tử/mẫu là số nguyên, không có sai số. Rất tiện để **kiểm chứng** lời giải dùng float.

In [6]:
x = Fraction(11, 2) * 15      # 5.5 * 15 viết bằng phân số
print(x, float(x))
print(Fraction(1, 10) + Fraction(2, 10) == Fraction(3, 10))

165/2 82.5
True


### 3.6 Chia số thực `/` và chia nguyên `//`

`/` luôn cho `float` (`15 / 2 = 7.5`), còn `//` **làm tròn xuống** (`15 // 2 = 7`). Dùng nhầm `//` khi tính vị trí kim giờ sẽ làm mất phần `.5` — một lỗi rất hay gặp (xem mục 13).

In [7]:
m = 15
print("m / 2  =", m / 2)
print("m // 2 =", m // 2)

m / 2  = 7.5
m // 2 = 7


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 3**: `3:15`.

- Kim phút: `15 phút × 6° = 90°` (chỉ đúng vào số 3).
- Kim giờ: nếu nghĩ "đúng 3 giờ" thì là `3 × 30° = 90°` ⇒ góc 0° — **sai!** Thực tế sau 15 phút, kim giờ đã nhích thêm `15 × 0.5° = 7.5°` về phía số 4 ⇒ kim giờ ở `97.5°`.
- Hiệu: `|97.5 − 90| = 7.5°`; cung còn lại `352.5°` ⇒ góc nhỏ hơn là **7.5°** ✔.

Tính tay **Ví dụ 1**: `12:30`.

- Kim phút: `30 × 6 = 180°`.
- Kim giờ: `12 % 12 = 0` ⇒ `0 × 30 + 30 × 0.5 = 15°`.
- Hiệu `165°`, cung còn lại `195°` ⇒ **165°** ✔.

| Thời điểm | Kim giờ (°) | Kim phút (°) | \|hiệu\| | 360 − hiệu | Đáp án |
|---|---|---|---|---|---|
| 12:30 | 15 | 180 | 165 | 195 | 165 |
| 3:30 | 105 | 180 | 75 | 285 | 75 |
| 3:15 | 97.5 | 90 | 7.5 | 352.5 | 7.5 |

In [8]:
def hand_positions(hour, minutes):
    hour_deg = (hour % 12) * 30 + minutes * 0.5
    minute_deg = minutes * 6
    return hour_deg, minute_deg

for h, m, want in examples:
    hd, md_ = hand_positions(h, m)
    d = abs(hd - md_)
    print(f"{h:2d}:{m:02d} | kim giờ={hd:6.1f} | kim phút={md_:6.1f} | hiệu={d:6.1f} | 360-hiệu={360 - d:6.1f} | đáp án={min(d, 360 - d)}")

12:30 | kim giờ=  15.0 | kim phút= 180.0 | hiệu= 165.0 | 360-hiệu= 195.0 | đáp án=165.0
 3:30 | kim giờ= 105.0 | kim phút= 180.0 | hiệu=  75.0 | 360-hiệu= 285.0 | đáp án=75.0
 3:15 | kim giờ=  97.5 | kim phút=  90.0 | hiệu=   7.5 | 360-hiệu= 352.5 | đáp án=7.5


## 5. Cách 1 — Brute Force / baseline: mô phỏng từng phút

**Ý tưởng:** đặt cả hai kim ở vạch 12 (0°) lúc `0:00`. Cho thời gian trôi **từng phút một**, mỗi phút cộng `6°` cho kim phút và `0.5°` cho kim giờ (lấy `% 360` để kim quay vòng). Sau đúng `(hour % 12) * 60 + minutes` phút, đọc vị trí hai kim và lấy góc nhỏ hơn.

Cách này không cần "nhìn ra" công thức — chỉ cần biết tốc độ quay. Để tránh mọi sai số, ta đo góc theo đơn vị **nửa độ** (half-degree): kim giờ đi `1` đơn vị/phút, kim phút đi `12` đơn vị/phút, vòng tròn = `720` đơn vị. Toàn số nguyên!

In [9]:
def angle_simulate(hour, minutes):
    """Mô phỏng từng phút, đơn vị nửa độ (720 đơn vị = 1 vòng)."""
    hour_pos = 0      # vị trí kim giờ (nửa độ)
    minute_pos = 0    # vị trí kim phút (nửa độ)
    total = (hour % 12) * 60 + minutes
    for _ in range(total):
        hour_pos = (hour_pos + 1) % 720      # 0.5 độ / phút
        minute_pos = (minute_pos + 12) % 720 # 6 độ / phút
    d = abs(hour_pos - minute_pos)
    return min(d, 720 - d) / 2               # đổi nửa độ -> độ

for h, m, want in examples:
    got = angle_simulate(h, m)
    print(f"{h:2d}:{m:02d} -> {got} (mong đợi {want})", "✔" if math.isclose(got, want) else "✘")

12:30 -> 165.0 (mong đợi 165.0) ✔
 3:30 -> 75.0 (mong đợi 75.0) ✔
 3:15 -> 7.5 (mong đợi 7.5) ✔


**Độ phức tạp:** số vòng lặp tối đa là `11 × 60 + 59 = 719` ⇒ `O(720)` = `O(1)` về mặt lý thuyết (vì input bị chặn), nhưng nếu tổng quát hóa thành "sau `T` phút" thì là `O(T)`. Bộ nhớ `O(1)`.

**Nút thắt:** ta đang "đi bộ" từng phút trong khi chuyển động của kim là **tuyến tính** theo thời gian — vị trí sau `T` phút chỉ là `tốc độ × T`. Không cần lặp, nhân thẳng là xong.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** chuyển động đều ⇒ vị trí = tốc độ × thời gian.

- Kim phút: `M = 6 · minutes`.
- Kim giờ: `H = 30 · (hour % 12) + 0.5 · minutes` (phần `0.5 · minutes` chính là "bẫy").
- Hiệu: `H − M = 30·h + 0.5·m − 6·m = 30·h − 5.5·m` (với `h = hour % 12`).

**Quy tắc:**
- Nếu `d = |30·h − 5.5·m|` thì góc nhỏ hơn là `min(d, 360 − d)`.
- Vì `0 ≤ 30·h ≤ 330` và `0 ≤ 5.5·m ≤ 324.5` nên `d ∈ [0, 330]` ⇒ **luôn `< 360`**, không cần `% 360` thêm.
- Với `hour = 12`, nếu quên `% 12` thì `30·12 − 5.5·m ∈ [35.5, 360]` vẫn ≤ 360 và `min(d, 360 − d)` vẫn cho đúng (vì `360 ≡ 0`). Tuy vậy, dùng `% 12` giúp code rõ nghĩa "12 giờ = vạch 0" và an toàn khi sửa đổi công thức.

**Bất biến:** góc giữa hai kim chỉ phụ thuộc **hiệu** hai vị trí modulo 360; lấy `min(d, 360 − d)` cho cung ngắn hơn.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tính vị trí từng kim rồi lấy hiệu

**Giải thích từng đoạn:**
1. `h = hour % 12` để 12 giờ trùng vạch 0.
2. Vị trí kim giờ `30*h + 0.5*minutes`; vị trí kim phút `6*minutes`.
3. `d = abs(...)` rồi `min(d, 360 - d)`.

Đây là cách **dễ giải thích nhất** khi phỏng vấn: nói rõ từng kim ở đâu.

In [10]:
def angle_positions(hour, minutes):
    h = hour % 12
    hour_deg = 30 * h + 0.5 * minutes
    minute_deg = 6 * minutes
    d = abs(hour_deg - minute_deg)
    return min(d, 360 - d)

for h, m, want in examples:
    print(f"{h:2d}:{m:02d} -> {angle_positions(h, m)} (mong đợi {want})")

12:30 -> 165.0 (mong đợi 165.0)
 3:30 -> 75.0 (mong đợi 75.0)
 3:15 -> 7.5 (mong đợi 7.5)


Code nộp LeetCode (Cách 2):

In [11]:
class Solution:
    def angleClock(self, hour: int, minutes: int) -> float:
        hour_deg = 30 * (hour % 12) + 0.5 * minutes
        minute_deg = 6 * minutes
        d = abs(hour_deg - minute_deg)
        return min(d, 360 - d)

print(Solution().angleClock(3, 15))

7.5


**Độ phức tạp:** `O(1)` thời gian, `O(1)` bộ nhớ.

### 7.2 Cách 3 — Công thức gộp `|30h − 5.5m|`

Rút gọn đại số: `d = |30·h − 5.5·m|`. Chỉ một dòng tính toán. Đây là công thức "kinh điển" trong các bài toán đồng hồ ở sách toán phổ thông.

In [12]:
def angle_formula(hour, minutes):
    d = abs(30 * (hour % 12) - 5.5 * minutes)
    return min(d, 360 - d)

for h, m, want in examples:
    print(f"{h:2d}:{m:02d} -> 30h-5.5m = {30 * (h % 12) - 5.5 * m:7.1f} -> {angle_formula(h, m)}")

12:30 -> 30h-5.5m =  -165.0 -> 165.0
 3:30 -> 30h-5.5m =   -75.0 -> 75.0
 3:15 -> 30h-5.5m =     7.5 -> 7.5


Code nộp LeetCode (Cách 3):

In [13]:
class Solution:
    def angleClock(self, hour: int, minutes: int) -> float:
        d = abs(30 * (hour % 12) - 5.5 * minutes)
        return min(d, 360 - d)

print(Solution().angleClock(12, 30))

165.0


### 7.3 Cách 4 — Số nguyên tuyệt đối (đơn vị nửa độ)

Nhân công thức với 2: `2d = |60·h − 11·m|` là **số nguyên**. Làm toàn bộ bằng số nguyên rồi chia 2 ở cuối ⇒ không bao giờ có sai số trung gian. Vòng tròn khi đó là `720` đơn vị.

Mẹo này hữu ích khi gặp các biến thể có giây (đơn vị nhỏ hơn nữa) hoặc khi cần so sánh bằng nhau tuyệt đối (ví dụ "đếm số thời điểm hai kim vuông góc").

In [14]:
def angle_int(hour, minutes):
    d2 = abs(60 * (hour % 12) - 11 * minutes)   # gấp đôi góc, số nguyên
    return min(d2, 720 - d2) / 2

for h, m, want in examples:
    print(f"{h:2d}:{m:02d} -> 2d = {abs(60 * (h % 12) - 11 * m):4d} -> {angle_int(h, m)}")

12:30 -> 2d =  330 -> 165.0
 3:30 -> 2d =  150 -> 75.0
 3:15 -> 2d =   15 -> 7.5


Code nộp LeetCode (Cách 4):

In [15]:
class Solution:
    def angleClock(self, hour: int, minutes: int) -> float:
        d2 = abs(60 * (hour % 12) - 11 * minutes)
        return min(d2, 720 - d2) / 2

print(Solution().angleClock(3, 30))

75.0


### 7.4 Cách 5 — Phân số chính xác (dùng để kiểm chứng)

Dùng `Fraction` cho tốc độ kim giờ `1/2` độ/phút. Kết quả là phân số chính xác; đổi sang `float` khi trả về. Cách này chậm hơn một chút nhưng là "trọng tài" tốt khi viết test.

In [16]:
def angle_fraction(hour, minutes):
    hour_deg = 30 * (hour % 12) + Fraction(1, 2) * minutes
    minute_deg = Fraction(6) * minutes
    d = abs(hour_deg - minute_deg)
    return min(d, 360 - d)          # trả về Fraction

for h, m, want in examples:
    r = angle_fraction(h, m)
    print(f"{h:2d}:{m:02d} -> {r} = {float(r)}")

12:30 -> 165 = 165.0
 3:30 -> 75 = 75.0
 3:15 -> 15/2 = 7.5


## 8. TRACE từng bước

### 8.1 Trace công thức (Cách 2) trên `3:15`

| Bước | Biểu thức | Giá trị |
|---|---|---|
| 1 | `h = 3 % 12` | 3 |
| 2 | kim giờ `30·3 + 0.5·15` | 97.5 |
| 3 | kim phút `6·15` | 90 |
| 4 | `d = |97.5 − 90|` | 7.5 |
| 5 | `360 − d` | 352.5 |
| 6 | `min(7.5, 352.5)` | **7.5** |

In [17]:
def trace_formula(hour, minutes):
    h = hour % 12
    print(f"h = {hour} % 12 = {h}")
    hour_deg = 30 * h + 0.5 * minutes
    print(f"kim giờ  = 30*{h} + 0.5*{minutes} = {hour_deg}")
    minute_deg = 6 * minutes
    print(f"kim phút = 6*{minutes} = {minute_deg}")
    d = abs(hour_deg - minute_deg)
    print(f"d = |{hour_deg} - {minute_deg}| = {d}; 360 - d = {360 - d}")
    ans = min(d, 360 - d)
    print("=> đáp án:", ans)
    return ans

trace_formula(3, 15)
print()
trace_formula(12, 30)

h = 3 % 12 = 3
kim giờ  = 30*3 + 0.5*15 = 97.5
kim phút = 6*15 = 90
d = |97.5 - 90| = 7.5; 360 - d = 352.5
=> đáp án: 7.5

h = 12 % 12 = 0
kim giờ  = 30*0 + 0.5*30 = 15.0
kim phút = 6*30 = 180
d = |15.0 - 180| = 165.0; 360 - d = 195.0
=> đáp án: 165.0


165.0

### 8.2 Trace mô phỏng (Cách 1) trên `1:05`

Ta in trạng thái sau mỗi 5 phút (đơn vị nửa độ). Tổng thời gian `1·60 + 5 = 65` phút.

In [18]:
def trace_simulate(hour, minutes, every=5):
    hour_pos = minute_pos = 0
    total = (hour % 12) * 60 + minutes
    print(f"tổng số phút cần mô phỏng: {total}")
    print(f"{'phút':>4} | {'kim giờ(°)':>10} | {'kim phút(°)':>11}")
    for t in range(1, total + 1):
        hour_pos = (hour_pos + 1) % 720
        minute_pos = (minute_pos + 12) % 720
        if t % every == 0 or t == total:
            print(f"{t:4d} | {hour_pos / 2:10.1f} | {minute_pos / 2:11.1f}")
    d = abs(hour_pos - minute_pos)
    ans = min(d, 720 - d) / 2
    print("=> góc nhỏ hơn:", ans)
    return ans

trace_simulate(1, 5)

tổng số phút cần mô phỏng: 65
phút | kim giờ(°) | kim phút(°)
   5 |        2.5 |        30.0
  10 |        5.0 |        60.0
  15 |        7.5 |        90.0
  20 |       10.0 |       120.0
  25 |       12.5 |       150.0
  30 |       15.0 |       180.0
  35 |       17.5 |       210.0
  40 |       20.0 |       240.0
  45 |       22.5 |       270.0
  50 |       25.0 |       300.0
  55 |       27.5 |       330.0
  60 |       30.0 |         0.0
  65 |       32.5 |        30.0
=> góc nhỏ hơn: 2.5


2.5

Nhận xét trace: ở phút 60 kim phút quay về 0° còn kim giờ ở 30° — đúng "1 giờ". Ở phút 65 kim phút ở 30°, kim giờ ở 32.5° ⇒ góc 2.5°. Công thức: `|30·1 − 5.5·5| = |30 − 27.5| = 2.5` ✔.

## 9. Code nộp LeetCode

Chọn công thức gộp (Cách 3) — ngắn, rõ, `O(1)`. Kết quả là `float`; các góc đều là bội 0.5 nên chính xác tuyệt đối.

In [19]:
class Solution:
    def angleClock(self, hour: int, minutes: int) -> float:
        # Kim giờ: 30 độ/giờ + 0.5 độ/phút; kim phút: 6 độ/phút
        # Hiệu = 30*h + 0.5*m - 6*m = 30*h - 5.5*m
        d = abs(30 * (hour % 12) - 5.5 * minutes)
        return min(d, 360 - d)

sol = Solution()
for h, m, want in examples:
    got = sol.angleClock(h, m)
    assert abs(got - want) < 1e-5, (h, m, got, want)
    print(f"angleClock({h}, {m}) = {got}")

angleClock(12, 30) = 165.0
angleClock(3, 30) = 75.0
angleClock(3, 15) = 7.5


## 10. Vì sao thuật toán đúng

1. **Vị trí kim phút:** kim phút quay đều 360° trong 60 phút, xuất phát từ 0° ở phút 0 ⇒ sau `m` phút ở `6m` độ.
2. **Vị trí kim giờ:** kim giờ quay đều 360° trong 720 phút. Tại `h:m` (với `h = hour % 12`), đã trôi qua `60h + m` phút kể từ 12:00 ⇒ vị trí `0.5·(60h + m) = 30h + 0.5m`.
3. **Góc giữa hai tia** từ tâm: hai cung tạo bởi hai tia có tổng 360°; một cung có số đo `d = |H − M|` (khi `0 ≤ d ≤ 360`), cung còn lại `360 − d`. Góc nhỏ hơn là `min(d, 360 − d)`.
4. **`d` nằm trong `[0, 360]`:** `H ∈ [0, 359.5]`, `M ∈ [0, 354]` ⇒ `|H − M| < 360`. Vậy bước 3 áp dụng được trực tiếp.

Mô phỏng (Cách 1) đúng vì nó cộng dồn đúng tốc độ đó từng phút; công thức chỉ là kết quả cộng dồn rút gọn. Cách 4 nhân 2 mọi vị trí — phép nhân với hằng dương không đổi thứ tự hay tỷ lệ — rồi chia 2 lại.

## 11. Phân tích Time / Space

Gọi `T = (hour % 12)·60 + minutes` là số phút từ 12:00 (≤ 719).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Mô phỏng | `O(T)` (≤ 719 bước) | `O(1)` | về lý thuyết là hằng số vì `T` bị chặn |
| 2. Vị trí từng kim | `O(1)` | `O(1)` | vài phép nhân/cộng |
| 3. Công thức gộp | `O(1)` | `O(1)` | 1 biểu thức |
| 4. Số nguyên nửa độ | `O(1)` | `O(1)` | không có sai số trung gian |
| 5. Fraction | `O(1)` | `O(1)` | hằng số lớn hơn (tạo đối tượng Fraction) |

Không có trường hợp average/worst khác nhau: mọi cách công thức đều chạy cùng một số phép tính cho mọi input.

## 12. Bộ test

### 12.1 Ví dụ của đề + một số thời điểm "đẹp"

In [20]:
METHODS = {
    "simulate": angle_simulate,
    "positions": angle_positions,
    "formula": angle_formula,
    "int": angle_int,
    "fraction": lambda h, m: float(angle_fraction(h, m)),
    "submit": Solution().angleClock,
}

special = examples + [
    (12, 0, 0.0),     # hai kim trùng nhau tại vạch 12
    (6, 0, 180.0),    # thẳng hàng
    (3, 0, 90.0),     # vuông góc
    (9, 0, 90.0),     # vuông góc phía bên kia
    (1, 0, 30.0),
    (11, 59, 5.5),    # sát 12 giờ: kim giờ 359.5, kim phút 354
    (4, 50, 155.0),
    (1, 57, 76.5),
]
for h, m, want in special:
    for name, f in METHODS.items():
        got = f(h, m)
        assert abs(got - want) < 1e-5, (name, h, m, got, want)
print(f"{len(special)} ca đặc biệt × {len(METHODS)} cách: tất cả đúng")

11 ca đặc biệt × 6 cách: tất cả đúng


### 12.2 Kiểm tra TOÀN BỘ 720 input

Vì không gian input nhỏ, ta đối chiếu mọi cách với "trọng tài" `Fraction` trên tất cả `hour ∈ [1,12]`, `minutes ∈ [0,59]`. Thêm kiểm tra tính chất: kết quả nằm trong `[0, 180]` và là bội của `0.5`.

In [21]:
count = 0
for h in range(1, 13):
    for m in range(60):
        ref = angle_fraction(h, m)
        assert 0 <= ref <= 180
        assert (ref * 2).denominator == 1          # bội của 0.5
        for name, f in METHODS.items():
            got = f(h, m)
            assert abs(got - float(ref)) < 1e-9, (name, h, m, got, ref)
        count += 1
print("Đã kiểm tra", count, "input × ", len(METHODS), "cách: khớp hoàn toàn")

Đã kiểm tra 720 input ×  6 cách: khớp hoàn toàn


### 12.3 Tính chất đối xứng (random có seed cố định)

Góc tại `h:m` bằng góc tại thời điểm **đối xứng qua trục 12–6**: nếu tổng phút từ 12:00 là `T` thì thời điểm `720 − T` (soi gương) cho cùng góc. Ta kiểm tra ngẫu nhiên tính chất đó.

In [22]:
rng = random.Random(1344)
for _ in range(300):
    h = rng.randint(1, 12)
    m = rng.randint(0, 59)
    T = (h % 12) * 60 + m
    T2 = (720 - T) % 720
    h2, m2 = divmod(T2, 60)
    h2 = 12 if h2 == 0 else h2
    assert angle_formula(h, m) == angle_formula(h2, m2), (h, m, h2, m2)
print("Tính chất soi gương đúng trên 300 bộ ngẫu nhiên (seed=1344)")
print("Ví dụ: 2:20 ->", angle_formula(2, 20), "| 9:40 ->", angle_formula(9, 40))

Tính chất soi gương đúng trên 300 bộ ngẫu nhiên (seed=1344)
Ví dụ: 2:20 -> 50.0 | 9:40 -> 50.0


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên rằng kim giờ cũng di chuyển theo phút

**Code sai:** coi kim giờ đứng yên ở vạch `30·h`.

In [23]:
def wrong_static_hour(hour, minutes):
    d = abs(30 * (hour % 12) - 6 * minutes)     # thiếu + 0.5*minutes
    return min(d, 360 - d)

for h, m, want in examples:
    got = wrong_static_hour(h, m)
    print(f"{h:2d}:{m:02d} -> sai cho {got}, đúng là {want}", "✘" if got != want else "")

12:30 -> sai cho 180, đúng là 165.0 ✘
 3:30 -> sai cho 90, đúng là 75.0 ✘
 3:15 -> sai cho 0, đúng là 7.5 ✘


**Vì sao sai:** lúc 3:15 kim giờ đã nhích 7.5° về phía số 4, nên hai kim **không** trùng nhau (0°). **Sửa:** kim giờ ở `30·h + 0.5·minutes` (xem Cách 2).

### Lỗi 2 — Không lấy góc nhỏ hơn

**Code sai:** trả về luôn `|H − M|`.

In [24]:
def wrong_no_min(hour, minutes):
    return abs(30 * (hour % 12) - 5.5 * minutes)

for h, m in [(12, 30), (9, 0), (10, 50)]:
    print(f"{h:2d}:{m:02d} -> sai cho {wrong_no_min(h, m)}, đúng là {angle_formula(h, m)}")

12:30 -> sai cho 165.0, đúng là 165.0
 9:00 -> sai cho 270.0, đúng là 90.0
10:50 -> sai cho 25.0, đúng là 25.0


**Vì sao sai:** `|H − M|` có thể lớn hơn 180 (ví dụ 9:00 cho 270°), đó là **cung lớn**. **Sửa:** `min(d, 360 − d)`.

### Lỗi 3 — Dùng chia nguyên `//` khi tính phần nhích của kim giờ

In [25]:
def wrong_floor_div(hour, minutes):
    hour_deg = 30 * (hour % 12) + minutes // 2   # mất phần .5 khi minutes lẻ
    d = abs(hour_deg - 6 * minutes)
    return min(d, 360 - d)

for h, m in [(3, 15), (1, 57), (11, 59)]:
    print(f"{h:2d}:{m:02d} -> sai cho {wrong_floor_div(h, m)}, đúng là {angle_formula(h, m)}")

 3:15 -> sai cho 7, đúng là 7.5
 1:57 -> sai cho 76, đúng là 76.5
11:59 -> sai cho 5, đúng là 5.5


**Vì sao sai:** `15 // 2 = 7` chứ không phải `7.5`. **Sửa:** dùng `minutes / 2` hoặc `0.5 * minutes`, hoặc làm toàn bộ theo đơn vị nửa độ (Cách 4).

### Lỗi 4 — Quên `abs`, kết quả âm

In [26]:
def wrong_no_abs(hour, minutes):
    d = 30 * (hour % 12) - 5.5 * minutes          # có thể âm
    return min(d, 360 - d)

for h, m in [(3, 30), (1, 30)]:
    print(f"{h:2d}:{m:02d} -> sai cho {wrong_no_abs(h, m)}, đúng là {angle_formula(h, m)}")

 3:30 -> sai cho -75.0, đúng là 75.0
 1:30 -> sai cho -135.0, đúng là 135.0


**Vì sao sai:** khi kim phút "vượt" kim giờ, hiệu âm; `min` chọn số âm. **Sửa:** `d = abs(...)`.

### Lỗi 5 — So sánh float bằng `==` trong test

Ở bài này may mắn đúng, nhưng nếu tính theo cách khác (ví dụ `minutes / 60 * 360`) sai số có thể xuất hiện. Luôn so sánh `abs(a − b) < 1e-5`.

In [27]:
a = 0.1 * 3 * 60          # một cách tính "lòng vòng"
b = 18.0
print("a == b ?", a == b, "| a =", repr(a))
print("abs(a-b) < 1e-5 ?", abs(a - b) < 1e-5)

a == b ? False | a = 18.000000000000004
abs(a-b) < 1e-5 ? True


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Bài đồng hồ** ⇒ nhớ 3 con số: `30°/giờ`, `6°/phút` (kim phút), `0.5°/phút` (kim giờ). Công thức: `|30h − 5.5m|`, rồi `min(d, 360 − d)`.
- **"Góc nhỏ hơn giữa hai hướng"** trên vòng tròn ⇒ luôn là `min(d, 360 − d)` sau khi đưa `d` về `[0, 360)`.
- Gặp số lẻ `.5` ⇒ cân nhắc **nhân đôi đơn vị** để làm số nguyên.
- Không gian input nhỏ (ở đây 720) ⇒ **test toàn bộ** thay vì vài ví dụ.
- Bài "Math" Medium trên LeetCode thường có bẫy nhỏ (ở đây là kim giờ di chuyển) — đọc kỹ ví dụ có giá trị lẻ như `7.5`.

## 15. Biến thể của bài

1. **Có thêm giây** `h:m:s`: kim giây `6°/giây`, kim phút `0.1°/giây`, kim giờ `1/120 °/giây`. Góc giữa kim giờ và kim phút: `|30h + m/2 + s/120 − 6m − s/10|`.
2. **Đếm số lần hai kim trùng nhau trong 12 giờ** (đáp án 11) hoặc vuông góc (22 lần).
3. **Tìm thời điểm sớm nhất sau `h:m` mà hai kim trùng nhau**: giải `30h − 5.5m ≡ 0 (mod 360)` theo `m` thực.
4. **Đồng hồ 24 giờ** (kim giờ quay 1 vòng/24h): tốc độ kim giờ thành `0.25°/phút`.
5. **Góc tới một hướng cố định** (ví dụ góc từ kim phút đến vạch 3) — cùng kỹ thuật `min(d, 360 − d)`.

In [28]:
def angle_with_seconds(h, m, s):
    """Biến thể 1: góc kim giờ - kim phút có tính giây (dùng Fraction cho chính xác)."""
    hour_deg = 30 * (h % 12) + Fraction(m, 2) + Fraction(s, 120)
    minute_deg = 6 * m + Fraction(s, 10)
    d = abs(hour_deg - minute_deg)
    return min(d, 360 - d)

print("3:15:00 ->", float(angle_with_seconds(3, 15, 0)))
print("3:15:30 ->", float(angle_with_seconds(3, 15, 30)))

# Biến thể 2: đếm số thời điểm (theo phút nguyên) hai kim cách nhau đúng 90 độ trong 12 giờ
right = [(h, m) for h in range(1, 13) for m in range(60) if angle_int(h, m) == 90]
print("Số phút nguyên có góc đúng 90°:", len(right), right)

3:15:00 -> 7.5
3:15:30 -> 4.75
Số phút nguyên có góc đúng 90°: 2 [(3, 0), (9, 0)]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng từng phút | cộng dồn tốc độ mỗi phút | `O(T)` ≤ 719 | `O(1)` | khi chưa nghĩ ra công thức; làm "trọng tài" |
| 2. Vị trí từng kim | tính `H`, `M` riêng rồi trừ | `O(1)` | `O(1)` | phỏng vấn, giải thích rõ ràng |
| 3. Công thức gộp | `|30h − 5.5m|` | `O(1)` | `O(1)` | nộp bài nhanh nhất |
| 4. Số nguyên nửa độ | `|60h − 11m| / 2` | `O(1)` | `O(1)` | cần so sánh chính xác, biến thể đếm |
| 5. Fraction | phân số chính xác | `O(1)` | `O(1)` | kiểm chứng, biến thể có giây |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** Không chạy code, đoán `angle_formula(6, 30)` và `angle_formula(9, 45)`.

**Bài 2 (tính tay):** Tính bằng tay góc nhỏ hơn lúc `7:20`. Ghi rõ vị trí kim giờ, kim phút.

**Bài 3 (điền TODO):** Hoàn thiện hàm `angle_24h(hour, minutes)` cho đồng hồ có **24 vạch giờ** (kim giờ quay 1 vòng trong 24 giờ, `0 <= hour <= 23`).

**Bài 4 (tự viết lại):** Viết hàm `count_overlaps()` đếm số **phút nguyên** trong 12 giờ (từ 12:00 đến 11:59) mà hai kim cách nhau **không quá 3°**.

**Bài 5 (mini challenge):** Viết hàm `next_overlap(hour, minutes)` trả về số **phút (Fraction)** cần chờ tính từ `hour:minutes` đến lần **kế tiếp** (hoặc ngay lúc đó) hai kim trùng nhau.

In [29]:
# Bài 3 — điền TODO (chỉ định nghĩa, không gọi)
def angle_24h(hour, minutes):
    # TODO: kim giờ quay 360 độ trong 24*60 phút -> bao nhiêu độ mỗi phút?
    # TODO: tính vị trí kim giờ, kim phút, rồi lấy góc nhỏ hơn
    pass

In [30]:
# Bài 4 — khung để tự viết (chỉ định nghĩa, không gọi)
def count_overlaps():
    # TODO: duyệt mọi (h, m) trong 12 giờ, đếm số lần góc <= 3
    pass

# Bài 5 — khung để tự viết (chỉ định nghĩa, không gọi)
def next_overlap(hour, minutes):
    # TODO: kim phút đuổi kim giờ với vận tốc tương đối 5.5 độ/phút
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1:** (1) Áp dụng `|30h − 5.5m|`. (2) `6:30`: `|180 − 165|`. (3) `9:45`: `|270 − 247.5|`.

**Bài 2:** (1) Kim phút `20 × 6`. (2) Kim giờ `7 × 30 + 20 × 0.5`. (3) Hiệu là 100°, so với `360 − 100`.

**Bài 3:** (1) Kim giờ đi `360 / 1440 = 0.25°/phút`, tức `15°/giờ`. (2) Kim phút vẫn `6°/phút`. (3) `d = |15·(hour % 24) + 0.25·m − 6m|`, rồi `min(d, 360 − d)`.

**Bài 4:** (1) Hai vòng `for`. (2) Dùng `angle_int` để tránh sai số. (3) Điều kiện `angle <= 3` — kiểm tra kết quả bằng cách in danh sách các thời điểm.

**Bài 5:** (1) Hai kim trùng nhau khi kim phút "đuổi kịp" kim giờ. (2) Khoảng cách theo chiều quay từ kim phút tới kim giờ là `g = (H − M) mod 360`. (3) Vận tốc tương đối `11/2 °/phút` ⇒ thời gian chờ `g / (11/2)`.

## 19. Đáp án tham khảo cho bài tập

In [31]:
# Bài 1
print("Bài 1:", angle_formula(6, 30), angle_formula(9, 45))
assert angle_formula(6, 30) == 15.0
assert angle_formula(9, 45) == 22.5

# Bài 2: 7:20 -> kim giờ 220, kim phút 120, hiệu 100, góc nhỏ hơn 100
hour_deg, minute_deg = 7 * 30 + 20 * 0.5, 20 * 6
print("Bài 2:", hour_deg, minute_deg, min(abs(hour_deg - minute_deg), 360 - abs(hour_deg - minute_deg)))
assert angle_formula(7, 20) == 100.0

Bài 1: 15.0 22.5
Bài 2: 220.0 120 100.0


In [32]:
# Bài 3
def angle_24h_sol(hour, minutes):
    hour_deg = 15 * (hour % 24) + 0.25 * minutes
    minute_deg = 6 * minutes
    d = abs(hour_deg - minute_deg)
    return min(d, 360 - d)

print("Bài 3:", angle_24h_sol(12, 0), angle_24h_sol(6, 0), angle_24h_sol(18, 30))
assert angle_24h_sol(12, 0) == 180.0     # kim giờ chỉ xuống dưới, kim phút ở 0
assert angle_24h_sol(0, 0) == 0.0
assert angle_24h_sol(6, 0) == 90.0
assert angle_24h_sol(18, 30) == 90.0 + 7.5  # kim giờ 277.5, kim phút 180 -> 97.5

Bài 3: 180.0 90.0 97.5


In [33]:
# Bài 4
def count_overlaps_sol(limit=3):
    res = []
    for h in range(1, 13):
        for m in range(60):
            if angle_int(h, m) <= limit:
                res.append((h, m))
    return res

near = count_overlaps_sol()
print("Bài 4:", len(near), near)
assert (12, 0) in near and (1, 5) in near
assert all(angle_fraction(h, m) <= 3 for h, m in near)
assert len(near) == sum(1 for h in range(1, 13) for m in range(60) if angle_fraction(h, m) <= 3)

Bài 4: 13 [(1, 5), (1, 6), (2, 11), (3, 16), (4, 22), (5, 27), (6, 33), (7, 38), (8, 44), (9, 49), (10, 54), (10, 55), (12, 0)]


In [34]:
# Bài 5
def next_overlap_sol(hour, minutes):
    H = 30 * (hour % 12) + Fraction(minutes, 2)
    M = Fraction(6 * minutes)
    gap = (H - M) % 360                 # kim phút phải quay thêm bao nhiêu độ (tương đối)
    return gap / Fraction(11, 2)        # phút

w = next_overlap_sol(3, 0)
print("Bài 5: từ 3:00 chờ", w, "phút ≈", float(w))
assert next_overlap_sol(12, 0) == 0
assert next_overlap_sol(3, 0) == Fraction(180, 11)      # ~16.36 phút -> 3:16:22
# kiểm tra: sau thời gian chờ, hai kim trùng nhau
for h in range(1, 13):
    for m in range(0, 60, 7):
        t = next_overlap_sol(h, m)
        T = (h % 12) * 60 + m + t
        assert (T * Fraction(1, 2) - T * 6) % 360 == 0
print("Bài 5: mọi thời gian chờ đều làm hai kim trùng nhau ✔")

Bài 5: từ 3:00 chờ 180/11 phút ≈ 16.363636363636363
Bài 5: mọi thời gian chờ đều làm hai kim trùng nhau ✔


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
class Solution:
    def angleClock(self, hour: int, minutes: int) -> float:
        d = abs(30 * (hour % 12) - 5.5 * minutes)   # kim giờ 30h+0.5m, kim phút 6m
        return min(d, 360 - d)                       # cung nhỏ hơn
```

Phiên bản số nguyên: `d2 = abs(60*(hour%12) - 11*minutes); return min(d2, 720 - d2) / 2`.

**Checklist trước khi submit:**
- [ ] Kim giờ có cộng `0.5 * minutes` (thử `3:15` phải ra `7.5`).
- [ ] Có `abs` và `min(d, 360 − d)` (thử `9:00` phải ra `90`, `12:30` phải ra `165`).
- [ ] Không dùng `//` cho phần nửa độ.
- [ ] `hour = 12` xử lý đúng (`12:00` ra `0`).
- [ ] Kết quả nằm trong `[0, 180]`.
- [ ] Đúng chữ ký: `angleClock(self, hour: int, minutes: int) -> float`.